In [1]:
import binascii
import serial
import time
import random
from fastecdsa.curve import Curve
from fastecdsa.point import Point
import sys
import os
import numpy as np
from tqdm.notebook import tqdm
import serial
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [2]:
%matplotlib inline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display, HTML

def analyze_and_plot_faults(log_file="results.csv", totals_file="results_total.csv"):
    if not os.path.exists(log_file):
        display(HTML(f"<b style='color: #e74c3c;'>Error: '{log_file}' not found. Run a successful attack first!</b>"))
        return

    # --- CONFIGURATION VARIABLES ---
    n_top = 40
    min_hits = 6 
    min_prob = 70.0  # Minimum predictability threshold for the heatmaps
    # -------------------------------
    
    df = pd.read_csv(log_file)
    
    if df.empty:
        display(HTML("<b style='color: #f39c12;'>The log file is empty. No faults recorded yet.</b>"))
        return
        
    # --- CALCULATE TOTAL FAULTS & TOTAL EXECUTIONS ---
    total_injected = len(df)
    total_executions = "Unknown"
    if os.path.exists(totals_file):
        try:
            df_totals = pd.read_csv(totals_file)
            if not df_totals.empty and 'total_executions' in df_totals.columns:
                total_executions = f"{df_totals['total_executions'].sum():,}"
        except Exception as e:
            display(HTML(f"<b style='color: #e67e22;'>Warning: Could not read {totals_file}: {e}</b>"))

    # ==========================================
    # Part 1: Data Table of Top Faults & Percentages
    # ==========================================
    display(HTML(f"<h3>🏆 Top {n_top} Most Reliable Single Faults (Min {min_hits} Hits)</h3>"))
    display(HTML(f"<p><b>Campaign Summary:</b> {total_injected:,} successful faults out of {total_executions} total injections attempted.</p>"))
    display(HTML("<p><i>Percentage represents: Out of all faults triggered at this specific Delay/Pulse, how often was it this exact byte change?</i></p>"))
    
    fault_counts = df.groupby(['delay', 'pulse', 'byte_index', 'expected_val', 'faulty_val']).size().reset_index(name='specific_fault_hits')
    total_faults_per_param = df.groupby(['delay', 'pulse']).size().reset_index(name='total_faults_at_params')
    merged_stats = pd.merge(fault_counts, total_faults_per_param, on=['delay', 'pulse'])
    merged_stats['probability (%)'] = (merged_stats['specific_fault_hits'] / merged_stats['total_faults_at_params']) * 100
    merged_stats['probability (%)'] = merged_stats['probability (%)'].round(2)
    
    filtered_stats = merged_stats[merged_stats['specific_fault_hits'] >= min_hits]
    
    top_faults = filtered_stats.sort_values(
        by=['probability (%)', 'specific_fault_hits'], 
        ascending=[False, False]
    ).head(n_top).reset_index(drop=True)
    
    if top_faults.empty:
        display(HTML(f"<b style='color: #e74c3c;'>No faults found with at least {min_hits} hits yet.</b>"))
    else:
        display(top_faults)

    # ==========================================
    # Part 2: Heatmap 1 (Strict Single-Value Reliability)
    # ==========================================
    display(HTML(f"<hr><h3>🔥 Heatmap 1: &ge;{min_prob}% Single-Value Reliability</h3>"))
    
    perfect_faults = filtered_stats[filtered_stats['probability (%)'] >= min_prob]
    df_heatmap1 = df.merge(
        perfect_faults[['delay', 'pulse', 'byte_index', 'expected_val', 'faulty_val', 'probability (%)']], 
        on=['delay', 'pulse', 'byte_index', 'expected_val', 'faulty_val'], 
        how='inner'
    )
    
    if df_heatmap1.empty:
        display(HTML(f"<p style='color: #e74c3c;'><b>No &ge;{min_prob}% reliable single faults found yet.</b></p>"))
    else:
        counts_pivot1 = df_heatmap1.pivot_table(index='byte_index', columns='delay', aggfunc='size', fill_value=0)
        prob_pivot1 = df_heatmap1.pivot_table(index='byte_index', columns='delay', values='probability (%)', aggfunc='mean', fill_value=0)
        
        annot_matrix1 = pd.DataFrame(index=counts_pivot1.index, columns=counts_pivot1.columns, dtype=str)
        for row in counts_pivot1.index:
            for col in counts_pivot1.columns:
                count = counts_pivot1.loc[row, col]
                if count > 0:
                    prob = prob_pivot1.loc[row, col]
                    annot_matrix1.loc[row, col] = f"{count}\n{prob:.0f}%"
                else:
                    annot_matrix1.loc[row, col] = "" 

        plt.figure(figsize=(15, 6))
        sns.heatmap(prob_pivot1, cmap="YlOrRd", annot=annot_matrix1, fmt="", 
                    linewidths=0.5, cbar_kws={'label': 'Single Fault Repeatability (%)'},
                    mask=(counts_pivot1 == 0), vmin=min_prob, vmax=100.0,
                    annot_kws={'size': 10}) # Slightly smaller text for fit
        
        plt.title(f"Single-Value Reliability (\u2265{min_prob}%, Min {min_hits} Hits)", fontsize=14, pad=15)
        plt.xlabel("Injection Delay (Cycles)", fontsize=12)
        plt.ylabel("Corrupted Byte Index (0 = MSB, 31 = LSB)", fontsize=12)
        plt.tight_layout()
        plt.show()

    # ==========================================
    # Part 3: Heatmap 2 (Low Variability / Top-2 Predictability)
    # ==========================================
    display(HTML(f"<hr><h3>🔥 Heatmap 2: &ge;{min_prob}% Low Variability (Top-2 Values)</h3>"))
    display(HTML("<p><i>Color represents the combined probability that a fault falls into the Top 1 or 2 most frequent values. Format inside cells: [%] / [Hits] / [Values]</i></p>"))

    heatmap2_records = []
    
    for (delay, pulse, byte_idx), group in df.groupby(['delay', 'pulse', 'byte_index']):
        total_hits = len(group)
        if (total_hits < min_hits):
            continue
            
        val_counts = group['faulty_val'].value_counts()
        unique_count = len(val_counts)
        top_vals = val_counts.index.tolist()
        
        # Determine Top-2 Hits and apply compact formatting
        if unique_count == 1:
            top_hits = val_counts.iloc[0]
            text_val = f"{top_vals[0]}"
        elif unique_count == 2:
            top_hits = val_counts.iloc[0] + val_counts.iloc[1]
            text_val = f"{top_vals[0]},{top_vals[1]}"
        else:
            top_hits = val_counts.iloc[0] + val_counts.iloc[1]
            text_val = f"{top_vals[0]},{top_vals[1]},+"
            
        predictability = (top_hits / total_hits) * 100.0
        
        if predictability >= min_prob:
            annot_text = f"{total_hits}\n{predictability:.0f}%"
            
            heatmap2_records.append({
                'delay': delay,
                'pulse': pulse,
                'byte_index': byte_idx,
                'hits': total_hits,
                'predictability': predictability,
                'annot_text': annot_text
            })

    df_heatmap2 = pd.DataFrame(heatmap2_records)
    
    if df_heatmap2.empty:
        display(HTML(f"<p style='color: #e74c3c;'><b>No faults meeting the >= {min_prob}% predictability criteria found.</b></p>"))
    else:
        counts_pivot2 = df_heatmap2.pivot_table(index='byte_index', columns='delay', values='hits', aggfunc='sum', fill_value=0)
        prob_pivot2 = df_heatmap2.pivot_table(index='byte_index', columns='delay', values='predictability', aggfunc='mean', fill_value=0)
        annot_pivot2 = df_heatmap2.pivot_table(index='byte_index', columns='delay', values='annot_text', aggfunc='first', fill_value="")

        plt.figure(figsize=(15, 6))
        
        sns.heatmap(prob_pivot2, cmap="YlOrRd", annot=annot_pivot2, fmt="", 
                    linewidths=0.8, cbar_kws={'label': 'Top-2 Predictability (%)'},
                    mask=(counts_pivot2 == 0), vmin=min_prob, vmax=100.0,
                    annot_kws={'size': 10}) 
        
        plt.title(f"Top-2 Predictability (\u2265{min_prob}%, Min {min_hits} Hits)", fontsize=14, pad=15)
        plt.xlabel("Injection Delay (Cycles)", fontsize=12)
        plt.ylabel("Corrupted Byte Index (0 = MSB, 31 = LSB)", fontsize=12)
        plt.tight_layout()
        plt.show()

        # ==========================================
        # Part 4: Extract Targeted Injection List
        # ==========================================
        extracted_settings_hm1 = df_heatmap1[['delay', 'pulse']].drop_duplicates().sort_values(by=['delay'])
        extracted_settings_hm2 = df_heatmap2[['delay', 'pulse']].drop_duplicates().sort_values(by=['delay'])
        settings_list_1 = list(extracted_settings_hm1.itertuples(index=False, name=None))
        settings_list_2 = list(extracted_settings_hm2.itertuples(index=False, name=None))
        settings_list = list(sorted(set(settings_list_1 + settings_list_2)))
        
        display(HTML("<hr><h3>🎯 Targeted Injection Settings</h3>"))
        display(HTML(f"<pre style='background-color: #1e1e1e; color: #a6e22e; padding: 10px; border-radius: 5px; font-family: monospace;'>{settings_list}</pre>"))

# Run the function
analyze_and_plot_faults(log_file="faults_log.csv", totals_file="faults_total.csv")
